In [1]:
# --- repo-root guard ---
# Notebooks live in notebooks/, but all data/code paths are relative to the repo root.
# If launched with the working directory set to notebooks/, step up one level so that
# 'src/', 'data/', 'pride_data/' and 'figures/' resolve correctly. Idempotent / no-op at root.
import os, pathlib
_cwd = pathlib.Path.cwd()
if not (_cwd / 'src').exists() and (_cwd.parent / 'src').exists():
    os.chdir(_cwd.parent)
print('working directory:', pathlib.Path.cwd())


working directory: D:\Projects\nanoPhos_env


In [2]:
# --- output directories (created relative to repo root) ---
from pathlib import Path as _P
for _d in ('figures','figures/figure2','figures/figure3','figures/figure4','figures/figure5'):
    _P(_d).mkdir(parents=True, exist_ok=True)

import sys, os
sys.path.insert(0, os.path.abspath("src"))
import re
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import importlib, core
importlib.reload(core)
from core import (count_sites_per_sample_ptm_report, process_ptm_site_report, _hex_to_rgba)

In [3]:
# Shared palettes (match Figure 3 v01)
color_sequence_red    = ['#FBA08D', '#FA7A61', '#F95534', '#ED2E07', '#CB2706', '#9E1E05']
color_sequence_violet = ['#C79EEA', '#B178E2', '#9B52DA', '#7E2AC7', '#6D25AD', '#551D87']
HELA_CELL_ORDER = [100, 300, 500, 1000, 2000, 3000]
STEM_CELL_ORDER = [100, 300, 500, 800, 1000, 3000]
PROC_META = {'Protein_group', 'Gene_group', 'PTM_0_aa', 'PTM_pos', 'PTM_mult123',
             'PTM_flank', 'PTM_Collapse_key', 'PTM_localization', 'UPD_seq'}

# Data upload

In [4]:
# Supplementary Figure 3 data = wide PTM Site Reports (per-run localization
# at 0.75), same format/source as Figure 3 (pride_data/analysis/figure3). Bucketed by
# workflow x system; the stem-cell SORTED files are multi-condition (cellline1..5).
RAW_DIR = Path('pride_data/analysis/figure3')

def parse_filename(fname):
    workflow = 'nanophos' if 'nanoPhos' in fname else ('uphos' if 'uPhos' in fname else 'unknown')
    if 'HeLa' in fname:
        system = 'hela'
    elif 'StemCells' in fname:
        system = 'stem'
    else:
        system = 'unknown'
    m = re.search(r'(\d+)cells', fname)
    return workflow, system, (int(m.group(1)) if m else None)

buckets = {}
for path in sorted(RAW_DIR.glob('[0-9]*_Report.tsv')):   # Class I site reports (timestamped exports)
    if 'RAlin' in path.name or 'normalized' in path.name:
        continue 
    w, s, n = parse_filename(path.name)
    if n is None:
        continue
    buckets.setdefault(f'{w}_{s}', {})[n] = path

loaded = {}
for group, files in buckets.items():
    loaded[group] = {n: pd.read_csv(files[n], sep='\t', low_memory=False) for n in sorted(files)}
    print(f'{group:<16} cells: {sorted(files)}')

hela_nanophos = loaded.get('nanophos_hela', {})
hela_uphos    = loaded.get('uphos_hela',    {})
stem_sorted   = loaded.get('nanophos_stem', {})

uphos_hela       cells: [100, 300, 500, 1000, 2000, 3000]


nanophos_hela    cells: [100, 300, 500, 1000, 2000, 3000]


nanophos_stem    cells: [100, 300, 500, 800, 1000, 3000]


# Supplementary Figure 3a

Phosphopeptide-precursor depth vs sorted-cell number (HeLa nanoPhos) — the identification-depth counterpart to the Class I depth in Fig. 3a, same boxplot convention. Phosphopeptide precursors = unique phosphorylated precursors (charge-distinct `EG.PrecursorId`), localization-independent, per Bekker-Jensen et al. 2020.

In [5]:
# Supplementary Figure 3a — phosphopeptide-precursor depth vs sorted-cell number (HeLa nanoPhos).
# Identification-depth counterpart to the Class I depth in Fig. 3a. Phosphopeptide precursors =
# unique phosphorylated precursors (charge-distinct EG.PrecursorId), localization-independent,
# per Bekker-Jensen et al. 2020. Counted per RUN, decoys removed (n=3 per cell number).
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from core import _hex_to_rgba

PREC_DIR = r'pride_data/analysis/figure3'
HELA_CELL_ORDER = [100, 300, 500, 1000, 2000, 3000]

def count_phosphoprecursors_per_run(path):
    """Unique phosphorylated precursors (EG.PrecursorId) per run; decoys removed."""
    df = pd.read_csv(path, sep='\t',
                     usecols=['R.FileName', 'EG.ModifiedSequence', 'EG.PrecursorId', 'EG.IsDecoy'],
                     low_memory=False)
    decoy = (df['EG.IsDecoy'].astype(str).str.lower().isin(['true', '1'])
             if df['EG.IsDecoy'].dtype != bool else df['EG.IsDecoy'])
    df = df[~decoy]
    df = df[df['EG.ModifiedSequence'].astype(str).str.contains('Phospho', case=False, na=False)]
    return df.groupby('R.FileName')['EG.PrecursorId'].nunique().tolist()

hela_precursors = {}   # {cell_number: [per-run counts]}
for n in HELA_CELL_ORDER:
    p = os.path.join(PREC_DIR, f'nanoPhos_new_HeLa_sorted_{n}cells_Report.tsv')
    if os.path.exists(p):
        hela_precursors[n] = count_phosphoprecursors_per_run(p)

print(f"{'cells':>6}  {'n':>3}  {'mean':>8}  {'SD':>6}  {'CV%':>5}")
for n in sorted(hela_precursors):
    c = np.array(hela_precursors[n]); sd = c.std(ddof=1) if len(c) > 1 else 0.0
    print(f"{n:>6}  {len(c):>3}  {int(round(c.mean())):>8,}  {int(round(sd)):>6,}  {100*sd/c.mean():>5.1f}")

BOX_COLOR, POINT_COLOR = '#1F5FA6', '#393E46'   # blue = phosphopeptide-precursor depth (as Suppl Fig 1b)
order = [n for n in HELA_CELL_ORDER if n in hela_precursors]
fig = go.Figure()
for n in order:
    ys = hela_precursors[n]
    fig.add_trace(go.Box(
        y=ys, x=[str(n)] * len(ys), name=str(n),
        boxpoints='all', jitter=0.3, pointpos=0,
        marker=dict(size=9, color=POINT_COLOR, line=dict(width=0.5, color='black')),
        line=dict(color=BOX_COLOR, width=1.5), fillcolor=_hex_to_rgba(BOX_COLOR, 0.15),
        showlegend=False))
fig.update_layout(template='plotly_white', width=600, height=600,
                  xaxis_title='Sorted cells', yaxis_title='Phosphopeptide precursors', showlegend=False)
fig.update_xaxes(categoryorder='array', categoryarray=[str(n) for n in order])
fig.update_yaxes(range = [0, 81000])
fig.show()
#fig.write_image(r'figures/figure3/suppl_figure3a.pdf', width=600, height=600)


 cells    n      mean      SD    CV%
   100    3    11,625     961    8.3
   300    3    29,826   4,111   13.8
   500    3    39,387   4,523   11.5
  1000    3    49,674   1,184    2.4
  2000    3    67,502     457    0.7
  3000    3    76,666     900    1.2


# Supplementary Figure 3b
Coefficient of variation of Class I phosphosite intensities across the 3 HeLa nanoPhos replicates, per sorted-cell number. Per project policy, CV is a per-feature quantitative metric — multiplicity is KEPT (no collapse).

In [6]:
# Suppl 3b - CV of Class I phosphosite intensities (HeLa nanoPhos), per cell number.
# Strict per-run Class I via process_ptm_site_report; CV computed in LINEAR space across
# the 3 replicates, requiring all 3 valid (matches Figure3_v00 logic). Multiplicity KEPT.
cv_by_cells = {}
for n in HELA_CELL_ORDER:
    if n not in hela_nanophos:
        continue
    sd = process_ptm_site_report(hela_nanophos[n], cutoff=0.75)['site_data']
    cols = [c for c in sd.columns if c not in PROC_META]
    lin = np.power(2.0, sd[cols])                       # log2 -> linear
    n_valid = lin.notna().sum(axis=1)
    cv = lin.std(axis=1) / lin.mean(axis=1)             # ddof=1 (pandas default), as in v00
    cv = cv.where(n_valid >= len(cols))                 # require all replicates valid
    cv_by_cells[n] = cv.dropna()

all_cv = np.concatenate([v.values for v in cv_by_cells.values()])
median_cv = float(np.nanmedian(all_cv))
print('median CV (all cell numbers): %.3f  (%.1f%%)' % (median_cv, 100 * median_cv))
for n, v in cv_by_cells.items():
    print(f'  {n:>4} cells: n={len(v):>5}  median CV={np.nanmedian(v):.3f}')

order = [n for n in HELA_CELL_ORDER if n in cv_by_cells]
fig = go.Figure()
for i, n in enumerate(order):
    fig.add_trace(go.Box(
        y=cv_by_cells[n], x=[str(n)] * len(cv_by_cells[n]), name=str(n),
        marker_color=color_sequence_red[i], line=dict(color=color_sequence_red[i]),
        boxpoints='outliers',                           # show outlier points beyond whiskers
        marker=dict(size=4, opacity=0.5), showlegend=False,
    ))
fig.add_hline(y=median_cv, line={'dash': 'dash', 'width': 2, 'color': 'black'})
fig.update_layout(width=600, height=600, template='plotly_white',
                  xaxis_title='Sorted cells', yaxis_title='Coefficient of variation')
fig.update_xaxes(categoryorder='array', categoryarray=[str(n) for n in order])
fig.update_yaxes(rangemode='tozero')
fig.show()
#fig.write_image(r'figures/figure3/suppl_figure3b.pdf', width=600, height=600)

Dropped 2,625 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 8,040 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 4,189 → 4,171.
Final: 4,171 sites × 3 samples.
Dropped 3,318 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 19,590 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 10,100 → 10,064.
Final: 10,064 sites × 3 samples.
Dropped 4,147 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 26,100 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 13,235 → 13,177.
Final: 13,177 sites × 3 samples.
Dropped 4,890 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']


Per-run mask (cutoff=0.75): 0 of 33,369 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 16,218 → 16,165.
Final: 16,165 sites × 3 samples.
Dropped 5,731 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 44,374 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 20,965 → 20,899.
Final: 20,899 sites × 3 samples.
Dropped 6,083 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 50,740 intensity cells masked (0.0%).


Deduplicated multi-protein rows: 23,797 → 23,726.
Final: 23,726 sites × 3 samples.
median CV (all cell numbers): 0.172  (17.2%)
   100 cells: n= 1452  median CV=0.287
   300 cells: n= 3362  median CV=0.309
   500 cells: n= 4712  median CV=0.270
  1000 cells: n= 6821  median CV=0.161
  2000 cells: n= 9550  median CV=0.149
  3000 cells: n=11014  median CV=0.130


# Supplementary Figure 3c

nanoPhos vs µPhos phosphopeptide-precursor advantage per sorted-cell number — the precursor-level analogue of Fig. 3c (Class I sites). Points: one per nanoPhos replicate divided by the µPhos mean (n = 3), mean ± SD crossbar; log y-axis, dotted line = parity. Identification depth isolates the enrichment contribution (upstream of localization).

In [7]:
# Supplementary Figure 3c — nanoPhos vs µPhos phosphopeptide-precursor advantage per sorted-cell number.
# Precursor-level analogue of Fig. 3c (Class I sites). Points: one per nanoPhos replicate / µPhos MEAN
# (n=3), mean ± SD crossbar, log y-axis. Reuses count_phosphoprecursors_per_run + hela_precursors (2a).
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

HELA_CELL_ORDER = [100, 300, 500, 1000, 2000, 3000]
PREC_DIR = r'pride_data/analysis/figure3'

uphos_precursors = {}
for n in HELA_CELL_ORDER:
    p = os.path.join(PREC_DIR, f'uPhos_HeLa_sorted_{n}cells_Report.tsv')
    if os.path.exists(p):
        uphos_precursors[n] = count_phosphoprecursors_per_run(p)

ratio_rows, ratios, labels = [], [], []
for n in HELA_CELL_ORDER:
    if n not in hela_precursors or n not in uphos_precursors:
        continue
    nano = hela_precursors[n]; um = float(np.mean(uphos_precursors[n]))
    if um <= 0:
        continue
    per_rep = [x / um for x in nano]
    ratios += per_rep; labels += [str(n)] * len(per_rep)
    ratio_rows.append({'cells': n, 'nano_mean': int(np.mean(nano)), 'uphos_mean': round(um, 1),
                       'ratio_of_means': round(np.mean(nano) / um, 1)})

ratio_summary = pd.DataFrame(ratio_rows).set_index('cells')
print(ratio_summary.to_string())

df_ratio = pd.DataFrame({'Ratio': ratios, 'ID': labels})
order = [str(n) for n in HELA_CELL_ORDER if n in hela_precursors and n in uphos_precursors]
fig = px.strip(df_ratio, y='Ratio', x='ID', log_y=True, category_orders={'ID': order})
fig.update_layout(width=600, height=600, template='plotly_white', showlegend=False,
                  xaxis_title='Sorted cells', yaxis_title='Ratio')
fig.update_traces(marker=dict(size=10, color='#1F5FA6', line=dict(width=0.5, color='black')))
fig.update_yaxes(showgrid=True, gridwidth=0.1, gridcolor='#F3F2F2', griddash='solid')
fig.update_xaxes(showgrid=True, gridwidth=0.1, gridcolor='#F3F2F2', griddash='solid')
fig.add_hline(y=1.0, line=dict(color='black', dash='dot', width=1.5))
stats_c = df_ratio.groupby('ID')['Ratio'].agg(['mean', 'std']).reindex(order)
fig.add_trace(go.Scatter(
    x=stats_c.index, y=stats_c['mean'],
    error_y=dict(type='data', array=stats_c['std'].fillna(0), visible=True,
                 color='black', thickness=1.5, width=10),
    mode='markers',
    marker=dict(symbol='line-ew', size=26, color='black', line=dict(width=1, color='black')),
    showlegend=False, hovertemplate='mean=%{y:.1f}x<extra></extra>'))
fig.show()
#fig.write_image(r'figures/figure3/suppl_figure3c.pdf', width=600, height=600)


       nano_mean  uphos_mean  ratio_of_means
cells                                       
100        11624      1283.0             9.1
300        29825      8600.0             3.5
500        39386     13661.3             2.9
1000       49674     18678.7             2.7
2000       67502     26984.7             2.5
3000       76665     26863.3             2.9


# Supplementary Figure 3d
Phosphopeptide selectivity (% phospho precursors) per sorted-cell number. Selectivity is a precursor-level enrichment-specificity metric, independent of Class I site localization — the original per-cell-number selectivity exports are reused unchanged.

In [8]:
# Suppl 3d - selectivity (REUSED original exports; not affected by Class I recount).
# Source: per-cell-number selectivity tables from the original figure3 analysis.
SEL_DIR = Path(r'pride_data/analysis/figure3')
sel_cell_order = [100, 300, 500, 1000, 2000, 3000]

sel = []
for n in sel_cell_order:
    df = pd.read_csv(SEL_DIR / f'selectivity_{n}cells.tsv', sep='\t')
    sel.append(df.set_index('XLabel').apply(np.sum, axis=1).tolist())

ids = np.repeat([str(n) for n in sel_cell_order], [len(s) for s in sel]).tolist()
df_sel = pd.DataFrame({'Selectivity': [v for s in sel for v in s], 'ID': ids})
print(df_sel.groupby('ID', sort=False)['Selectivity'].agg(['count', 'mean']).round(2).to_string())

fig = px.strip(df_sel, y='Selectivity', x='ID', orientation='h',
               category_orders={'ID': [str(n) for n in sel_cell_order]})
fig.update_layout(width=600, height=600, template='plotly_white', showlegend=False,
                  xaxis_title='Sorted cells', yaxis_title='Selectivity (%)')
fig.update_traces(marker=dict(size=18, color='#db4c2e', line=dict(width=0.5, color='black')))
fig.update_yaxes(range=[0, 100], showgrid=True, gridwidth=0.1, gridcolor='#F3F2F2')
fig.show()
# fig.write_image(r'figures/figure3/suppl_figure3d.pdf', width=600, height=600)

      count   mean
ID                
100       3  65.41
300       3  85.71
500       3  88.80
1000      3  88.18
2000      3  89.68
3000      3  88.96


# Supplementary Figure 3e
uPhos Class I phosphosite depth vs sorted-cell number (HeLa). Counterpart to Figure 3a (nanoPhos). Class I, multiplicity collapsed, localization enforced.

In [9]:
# Suppl 3e - uPhos HeLa Class I phosphosite depth vs cell number (box + points, n=3).
# Same hardened counter and styling as Figure 3a, applied to the uPhos arm.
counts_by_cells = {n: list(count_sites_per_sample_ptm_report(hela_uphos[n]).values())
                   for n in HELA_CELL_ORDER if n in hela_uphos}

rows = []
for n in HELA_CELL_ORDER:
    if n not in counts_by_cells:
        continue
    c = counts_by_cells[n]
    rows.append({'cells': n, 'n_reps': len(c), 'median': int(np.median(c)),
                 'mean': int(np.mean(c)),
                 'CV%': round(100 * np.std(c, ddof=1) / np.mean(c), 1) if len(c) > 1 else 0})
print(pd.DataFrame(rows).set_index('cells').to_string())

BOX_COLOR, POINT_COLOR = '#7E2AC7', '#393E46'     # violet box to distinguish from nanoPhos (3a)
order = [n for n in HELA_CELL_ORDER if n in counts_by_cells]
fig = go.Figure()
for n in order:
    ys = counts_by_cells[n]
    fig.add_trace(go.Box(
        y=ys, x=[str(n)] * len(ys), name=str(n),
        boxpoints='all', jitter=0.3, pointpos=0,
        marker=dict(size=9, color=POINT_COLOR, line=dict(width=0.5, color='black')),
        line=dict(color=BOX_COLOR, width=1.5), fillcolor=_hex_to_rgba(BOX_COLOR, 0.15),
        showlegend=False,
    ))
fig.update_layout(template='plotly_white', width=600, height=600, showlegend=False,
                  xaxis_title='Sorted cells', yaxis_title='Class I phosphosites (uPhos)')
fig.update_xaxes(categoryorder='array', categoryarray=[str(n) for n in order])
fig.update_yaxes(range = [0, 7100])
fig.show()
#fig.write_image(r'figures/figure3/suppl_figure3e.pdf', width=600, height=600)

       n_reps  median  mean   CV%
cells                            
100         3     569   501  24.6
300         3    2087  2073  11.0
500         3    3031  3178  11.5
1000        3    3936  3955   1.4
2000        3    5880  5865   1.0
3000        3    5844  6001   4.9


# Supplementary Figure 3f

Phosphopeptide-precursor depth per pluripotency state at 500 cells (normalized stem experiment) — the precursor-level analogue of Fig. 3f (Class I sites), same box convention and state colours (2iL/SL/RA). Phosphopeptide precursors = unique phosphorylated precursors (`EG.PrecursorId`), localization-independent.

In [10]:
# Supplementary Figure 3f — phosphopeptide-precursor depth per pluripotency state (500-cell
# normalized stem experiment). Precursor-level analogue of Fig. 3f (Class I). Phosphopeptide
# precursors = unique phosphorylated precursors (EG.PrecursorId), localization-independent.
import os, re
import numpy as np, pandas as pd
import plotly.graph_objects as go
from core import _hex_to_rgba

PREC_DIR = r'pride_data/analysis/figure3'
CELLLINE_TO_STATE = {1: '2iL', 2: 'SL', 4: 'RA'}     # cellline3 & 5(RA24) excluded (as Fig 3f)
STATE_ORDER = ['2iL', 'SL', 'RA']
STATE_COLOR = {'2iL': '#ED2E07', 'SL': '#7E2AC7', 'RA': '#2A88C7'}

def precursors_by_state(path):
    """{state: [per-run phosphoprecursor counts]}; state from cellline in R.FileName (fallback R.Condition)."""
    df = pd.read_csv(path, sep='\t',
                     usecols=['R.Condition', 'R.FileName', 'EG.ModifiedSequence', 'EG.PrecursorId', 'EG.IsDecoy'],
                     low_memory=False)
    decoy = (df['EG.IsDecoy'].astype(str).str.lower().isin(['true', '1'])
             if df['EG.IsDecoy'].dtype != bool else df['EG.IsDecoy'])
    df = df[~decoy]
    df = df[df['EG.ModifiedSequence'].astype(str).str.contains('Phospho', case=False, na=False)]
    per_run = df.groupby('R.FileName')['EG.PrecursorId'].nunique()
    run_cond = df.drop_duplicates('R.FileName').set_index('R.FileName')['R.Condition'].astype(str)
    by_state = {s: [] for s in STATE_ORDER}
    for run, cnt in per_run.items():
        m = re.search(r'cellline(\d+)', str(run))
        cl = int(m.group(1)) if m else (int(run_cond[run]) if run_cond[run].strip().isdigit() else None)
        st = CELLLINE_TO_STATE.get(cl)
        if st in by_state:
            by_state[st].append(int(cnt))
    return by_state

stem500_precursors = precursors_by_state(
    os.path.join(PREC_DIR, 'nanoPhos_StemCells_500cells_RAlin_SL_normalized_Report.tsv'))
for s in STATE_ORDER:
    c = stem500_precursors[s]
    if c:
        print(f'{s:<4} n={len(c)}  mean={int(np.mean(c))}  CV%={round(100*np.std(c, ddof=1)/np.mean(c), 1)}  vals={c}')

fig = go.Figure()
for s in STATE_ORDER:
    ys = stem500_precursors[s]
    if not ys:
        continue
    fig.add_trace(go.Box(
        y=ys, x=[s] * len(ys), name=s, boxpoints='all', jitter=0.3, pointpos=0,
        marker=dict(size=9, color=STATE_COLOR[s], line=dict(width=0.5, color='black')),
        line=dict(color=STATE_COLOR[s], width=1.5),
        fillcolor=_hex_to_rgba(STATE_COLOR[s], 0.2), showlegend=False))
fig.update_layout(width=500, height=600, template='plotly_white', showlegend=False,
                  xaxis_title='Pluripotency state (500 cells)', yaxis_title='Phosphopeptide precursors')
fig.update_xaxes(categoryorder='array', categoryarray=STATE_ORDER)
fig.update_yaxes(range = [0, 25100])
fig.show()
#fig.write_image(r'figures/figure3/suppl_figure3f.pdf', width=500, height=600)


2iL  n=3  mean=20579  CV%=5.6  vals=[21565, 19301, 20872]
SL   n=3  mean=20060  CV%=7.2  vals=[20492, 21234, 18454]
RA   n=3  mean=23392  CV%=1.6  vals=[23731, 22995, 23452]


# Supplementary Figure 3g
Per-state Class I phosphosite depth across sorted-cell numbers for the stem-cell sorted series. States 2iL/SL/RA (cellline 1/2/4), consistent with the rest of Figure 3 (cellline3 and cellline5=RA24 excluded). One separate plot per cell number.

In [11]:
# Suppl 3g - stem-cell Class I depth per state (2iL/SL/RA); ONE SEPARATE PLOT per cell number.
# 500 cells omitted here (shown in the main figure). Class I, multiplicity collapsed,
# localization enforced. cellline1=2iL, 2=SL, 4=RA.
CELLLINE_TO_STATE = {1: '2iL', 2: 'SL', 4: 'RA'}     # cellline3 & 5(RA24) excluded
STATE_ORDER = ['2iL', 'SL', 'RA']
STATE_COLOR = {'2iL': '#ED2E07', 'SL': '#7E2AC7', 'RA': '#2A88C7'}
STEM_PANELS = [n for n in STEM_CELL_ORDER if n != 500]   # 500 is in the main figure

def state_of(col):
    m = re.search(r'cellline(\d+)', col)
    return CELLLINE_TO_STATE.get(int(m.group(1))) if m else None

depth_by_cells = {}
for n in STEM_PANELS:
    if n not in stem_sorted:
        continue
    counts = count_sites_per_sample_ptm_report(stem_sorted[n])
    by_state = {s: [] for s in STATE_ORDER}
    for sample, c in counts.items():
        st = state_of(sample)
        if st in by_state:
            by_state[st].append(c)
    depth_by_cells[n] = by_state
    line = '  '.join(f'{s}: mean={int(np.mean(by_state[s]))}' for s in STATE_ORDER if by_state[s])
    print(f'{n:>4} cells | {line}')

# one independent figure per cell number; y-axis 0 -> 1.1 x (max point in that panel)
for n in [c for c in STEM_PANELS if c in depth_by_cells]:
    fig = go.Figure()
    panel_max = max(v for vals in depth_by_cells[n].values() for v in vals)
    for s in STATE_ORDER:
        ys = depth_by_cells[n][s]
        if not ys:
            continue
        fig.add_trace(go.Box(
            y=ys, x=[s] * len(ys), name=s, boxpoints='all', jitter=0.3, pointpos=0,
            marker=dict(size=9, color=STATE_COLOR[s], line=dict(width=0.5, color='black')),
            line=dict(color=STATE_COLOR[s], width=1.5),
            fillcolor=_hex_to_rgba(STATE_COLOR[s], 0.2), showlegend=False))
    fig.update_layout(width=500, height=500, template='plotly_white', showlegend=False,
                      title=f'{n} cells',
                      xaxis_title='Pluripotency state', yaxis_title='Class I phosphosites')
    fig.update_xaxes(categoryorder='array', categoryarray=STATE_ORDER)
    fig.update_yaxes(range=[0, panel_max * 1.2])         # 10% headroom above max point
    fig.show()
    #fig.write_image(rf'figures/figure3/suppl_figure3g_{n}cells.pdf', width=500, height=500)

 100 cells | 2iL: mean=1168  SL: mean=752  RA: mean=945
 300 cells | 2iL: mean=2342  SL: mean=2531  RA: mean=3103


 800 cells | 2iL: mean=5887  SL: mean=6311  RA: mean=7410


1000 cells | 2iL: mean=6906  SL: mean=6702  RA: mean=8391


3000 cells | 2iL: mean=11749  SL: mean=11916  RA: mean=13519


# Supplementary Figure 3h

Phosphopeptide-precursor depth per pluripotency state across sorted-cell numbers (stem dilution series) — the precursor-level analogue of the Class I stem panel (Suppl 2g); one plot per cell number, 500 cells omitted (shown in 2f).

In [12]:
# Supplementary Figure 3h — phosphopeptide-precursor depth per pluripotency state across sorted-cell
# numbers (stem dilution series). Precursor-level analogue of the Class I stem panel (Suppl 2g);
# ONE SEPARATE PLOT per cell number, 500 omitted (shown in 2f). Reuses precursors_by_state (2f).
import os
import numpy as np, pandas as pd
import plotly.graph_objects as go
from core import _hex_to_rgba

PREC_DIR = r'pride_data/analysis/figure3'
STEM_CELL_ORDER = [100, 300, 500, 800, 1000, 3000]
STEM_PANELS = [n for n in STEM_CELL_ORDER if n != 500]     # 500 shown in 2f
STATE_ORDER = ['2iL', 'SL', 'RA']
STATE_COLOR = {'2iL': '#ED2E07', 'SL': '#7E2AC7', 'RA': '#2A88C7'}

stem_precursors_by_cells = {}
for n in STEM_PANELS:
    p = os.path.join(PREC_DIR, f'nanoPhos_StemCells_{n}cells_Report.tsv')
    if not os.path.exists(p):
        continue
    by_state = precursors_by_state(p)
    stem_precursors_by_cells[n] = by_state
    line = '  '.join(f'{s}: mean={int(np.mean(by_state[s]))}' for s in STATE_ORDER if by_state[s])
    print(f'{n:>4} cells | {line}')

for n in [c for c in STEM_PANELS if c in stem_precursors_by_cells]:
    fig = go.Figure()
    panel_max = max(v for vals in stem_precursors_by_cells[n].values() for v in vals)
    for s in STATE_ORDER:
        ys = stem_precursors_by_cells[n][s]
        if not ys:
            continue
        fig.add_trace(go.Box(
            y=ys, x=[s] * len(ys), name=s, boxpoints='all', jitter=0.3, pointpos=0,
            marker=dict(size=9, color=STATE_COLOR[s], line=dict(width=0.5, color='black')),
            line=dict(color=STATE_COLOR[s], width=1.5),
            fillcolor=_hex_to_rgba(STATE_COLOR[s], 0.2), showlegend=False))
    fig.update_layout(width=500, height=500, template='plotly_white', showlegend=False,
                      title=f'{n} cells', xaxis_title='Pluripotency state',
                      yaxis_title='Phosphopeptide precursors')
    fig.update_xaxes(categoryorder='array', categoryarray=STATE_ORDER)
    fig.update_yaxes(range=[0, panel_max * 1.2])
    fig.show()
    #fig.write_image(rf'figures/figure3/suppl_figure3h_{n}cells.pdf', width=500, height=500)


 100 cells | 2iL: mean=4881  SL: mean=3367  RA: mean=4336


 300 cells | 2iL: mean=11605  SL: mean=12401  RA: mean=15282


 800 cells | 2iL: mean=27932  SL: mean=30281  RA: mean=36463


1000 cells | 2iL: mean=33829  SL: mean=33051  RA: mean=43504


3000 cells | 2iL: mean=58600  SL: mean=61203  RA: mean=71855


# Supplementary Figure 3i-l
DIA-NN vs Spectronaut on the sorted-cell HeLa dilution series. DIA-NN 2.2.0, one search per cell number against an in silico-predicted library, MBR on (logs: `pride_data/analysis/figure3/diann/HeLa_sorted_<n>cells/report.log.txt`). DIA-NN precursors are filtered (PG.Q.Value <= 0.05, Global.PG.Q.Value and Lib.PG.Q.Value <= 0.01, Quantity.Quality >= 0.5, PG.MaxLFQ.Quality >= 0.7), mapped onto the Spectronaut PSM schema and collapsed with `PeptideCollapse_v4` (per-run Class I >= 0.75), so both engines share the site key and localization rule.
- **3i** per-run Class I depth (multiplicity collapsed, Fig. 3a unit), n = 3 per cell number
- **3j** overlap of Class I site sets pooled across the 3 replicates: % of Spectronaut sites also found by DIA-NN, and Jaccard (shared / union)
- **3k-l** log2 intensity of shared vs engine-specific sites at 3,000 cells, each engine on its own scale (one-sided Mann-Whitney U, shared > engine-only)

In [13]:
# Supplementary Figure 3i-l — DIA-NN vs Spectronaut Class I phosphosites.
# DIA-NN 2.2.0 reports (one search per cell number, MBR on) are filtered (protein-group q-values +
# QuantUMS quality), mapped onto the Spectronaut PSM schema and collapsed with the repo's
# PeptideCollapse_v4 (per-run Class I >= 0.75), i.e. the same site key and localization rule as
# Spectronaut. Depth = per run, multiplicity collapsed (Fig. 3a unit); overlap = Class I site sets
# pooled across the 3 replicates per cell number.
from scipy import stats
from PeptideCollapse_v4 import PeptideCollapse

DIANN_DIR = Path('pride_data/analysis/figure3/diann')
CUTOFF = 0.75
_UNIMOD = re.compile(r'\(UniMod:\d+\)')
SITE_META = PROC_META | {'kinase_sequence', 'Protein_Collapse_key', 'PG.Genes', 'PG.ProteinGroups'}


def find_diann_report(n):
    # main report (MBR pass) of the search of the n-cell runs
    cands = list(DIANN_DIR.glob(f'HeLa_sorted_{n}cells/report.parquet'))
    if len(cands) != 1:
        raise FileNotFoundError(f'expected one DIA-NN main report for {n} cells, found {cands}')
    return cands[0]


def diann_precursor_id(modseq, charge):
    s = modseq.replace('(UniMod:21)', '[Phospho (STY)]')
    return '_' + _UNIMOD.sub('', s) + '_.' + str(int(charge))


def diann_loc_probs(occ):
    # 'AAS(UniMod:21){1.000000}LPT{0.848000}K2' -> '_AAS[Phospho (STY): 100%]LPT[Phospho (STY): 84.8%]K_'
    if not isinstance(occ, str):
        return np.nan
    s = re.sub(r'\d+$', '', _UNIMOD.sub('', occ))
    s = re.sub(r'\{([\d.]+)\}', lambda m: '[Phospho (STY): %.4g%%]' % (float(m.group(1)) * 100), s)
    return '_' + s + '_'


def diann_pep_start(modseq, protein_sites):
    # first S/T/Y position of the leading protein in Protein.Sites (skips bundled carbamidomethyl-C)
    inner = str(protein_sites).strip('[]').split(';')[0]
    sty = [t for t in inner.split(':', 1)[1].split(',') if t[:1] in 'STY'] if ':' in inner else []
    if not sty:
        return np.nan
    within = len(_UNIMOD.sub('', modseq[:modseq.index('(UniMod:21)')]))
    return int(re.sub(r'\D', '', sty[0])) - within + 1


def diann_classI(path):
    df = pd.read_parquet(path)
    df = df[df['Genes'].notna() & (df['Genes'].astype(str).str.strip() != '')]
    df = df[(df['PG.Q.Value'] <= 0.05) & (df['Global.PG.Q.Value'] <= 0.01) & (df['Lib.PG.Q.Value'] <= 0.01)
            & (df['Quantity.Quality'] >= 0.5) & (df['PG.MaxLFQ.Quality'] >= 0.7)]
    df = df[df['Modified.Sequence'].str.contains('UniMod:21', na=False)
            & df['Protein.Sites'].notna() & df['Site.Occupancy.Probabilities'].notna()]
    psm = pd.DataFrame({
        'R.FileName': df['Run'].astype(str),
        'EG.PrecursorId': [diann_precursor_id(m, c) for m, c in zip(df['Modified.Sequence'], df['Precursor.Charge'])],
        'EG.TotalQuantity (Settings)': df['Precursor.Quantity'].values,
        'PEP.PeptidePosition': [diann_pep_start(m, p) for m, p in zip(df['Modified.Sequence'], df['Protein.Sites'])],
        'EG.PTMAssayProbability': df['PTM.Site.Confidence'].values,
        'EG.PTMLocalizationProbabilities': [diann_loc_probs(o) for o in df['Site.Occupancy.Probabilities']],
        'PG.Genes': df['Genes'].astype(str).str.split(';').str[0],
        'PG.ProteinGroups': df['Protein.Group'].astype(str),
    }).dropna(subset=['PEP.PeptidePosition'])
    psm['PEP.PeptidePosition'] = psm['PEP.PeptidePosition'].astype(int)
    pc = PeptideCollapse(verbose=False)
    pc.process_complete_pipeline(psm, cutoff=CUTOFF, collapse_level='PG', aggregation_method='sum',
                                 return_both=False, add_kinase_sequences=False, noise_floor_filter=True,
                                 localization_strategy='per_run')
    return pc.site_data


def site_key(sd):
    # {Gene}_{aa}{pos}: multiplicity collapsed, protein group dropped (comparable across engines)
    return sd['PTM_Collapse_key'].str.split('~', n=1).str[1].str.replace(r'_M\d+$', '', regex=True)


def run_cols(sd):
    return [c for c in sd.columns if c not in SITE_META and pd.api.types.is_numeric_dtype(sd[c])]


def per_run_depth(sd):
    return sd[run_cols(sd)].notna().assign(_k=site_key(sd).values).groupby('_k').any().sum(axis=0).tolist()


def site_intensity(sd):
    # per Class I site (>= 1 run): max over multiplicities of the replicate-mean log2 intensity
    return pd.Series(sd[run_cols(sd)].mean(axis=1).values, index=site_key(sd).values).dropna().groupby(level=0).max()


engine_sites = {}
depth_rows, overlap_rows, abund_rows = [], [], []
for n in HELA_CELL_ORDER:
    sd_d = diann_classI(find_diann_report(n))
    sd_s = process_ptm_site_report(hela_nanophos[n], cutoff=CUTOFF)['site_data']
    for engine, sd in (('Spectronaut', sd_s), ('DIA-NN', sd_d)):
        for rep, v in enumerate(per_run_depth(sd), 1):
            depth_rows.append({'cells': n, 'engine': engine, 'replicate': rep, 'classI_sites': int(v)})
    Si, Di = site_intensity(sd_s), site_intensity(sd_d)
    engine_sites[n] = (Si, Di)
    S, D = set(Si.index), set(Di.index)
    overlap_rows.append({'cells': n, 'Spectronaut': len(S), 'DIA-NN': len(D), 'shared': len(S & D),
                         'pct_spectronaut_in_diann': round(100 * len(S & D) / len(S), 1),
                         'jaccard_pct': round(100 * len(S & D) / len(S | D), 1)})
    for engine, I, only in (('Spectronaut', Si, S - D), ('DIA-NN', Di, D - S)):
        sh, on = I[list(S & D)], I[list(only)]
        abund_rows.append({'cells': n, 'engine': engine, 'shared_median': round(sh.median(), 2),
                           'engine_only_median': round(on.median(), 2),
                           'p_mannwhitney_greater': stats.mannwhitneyu(sh, on, alternative='greater').pvalue})

depth_df = pd.DataFrame(depth_rows)
overlap_df = pd.DataFrame(overlap_rows)
abundance_df = pd.DataFrame(abund_rows)
depth_mean = depth_df.groupby(['cells', 'engine'])['classI_sites'].mean().unstack()
depth_mean['DIA-NN/Spectronaut %'] = (100 * depth_mean['DIA-NN'] / depth_mean['Spectronaut']).round(1)
print(depth_mean.round(1).to_string())
print(overlap_df.to_string(index=False))
print(abundance_df.to_string(index=False))

INFO	PeptideCollapse:PeptideCollapse_v4.py:process_complete_pipeline()- Starting complete pipeline


INFO	PeptideCollapse:PeptideCollapse_v4.py:load_data()- Loaded 15427 rows


INFO	PeptideCollapse:PeptideCollapse_v4.py:load_data()- Total rows loaded: 15427


INFO	PeptideCollapse:PeptideCollapse_v4.py:load_data()- Unique samples (R.FileName): 3


INFO	PeptideCollapse:PeptideCollapse_v4.py:preprocess_data()- Preprocessing: 15427 rows remaining after phospho filter, 0 non-phospho removed


INFO	PeptideCollapse:PeptideCollapse_v4.py:preprocess_data()- Preprocessing complete: 15427 phospho rows retained


... (116 outputs left out)

INFO	PeptideCollapse:PeptideCollapse_v4.py:_create_site_level_collapse()- Site final output: 20102 sites, 3 samples, completeness 82.4%, NaN 17.6%


INFO	PeptideCollapse:PeptideCollapse_v4.py:collapse_to_sites()- Site-level collapse complete: 20102 sites


INFO	PeptideCollapse:PeptideCollapse_v4.py:process_complete_pipeline()- Pipeline complete in 3.77 seconds


Dropped 6,083 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 50,740 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 23,797 → 23,726.
Final: 23,726 sites × 3 samples.
engine   DIA-NN  Spectronaut  DIA-NN/Spectronaut %
cells                                             
100      2616.7       2593.7                 100.9
300      6339.0       6240.0                 101.6
500      8487.0       8234.3                 103.1
1000    10241.3      10408.7                  98.4
2000    13637.3      13707.3                  99.5
3000    14403.0      15623.7                  92.2
 cells  Spectronaut  DIA-NN  shared  pct_spectronaut_in_diann  jaccard_pct
   100         4064    3576    2303                      56.7         43.2
   300         9625    8442    5767                      59.9         46.9
   500        12527   11276    7797                      62.2         48.7
  1

In [14]:
# Suppl 3i - per-run Class I depth vs sorted cells, Spectronaut vs DIA-NN (mean of the n=3 runs).
ENGINE_COLOR = {'Spectronaut': 'darkred', 'DIA-NN': '#2b8cbe'}
ENGINE_LABEL = {'Spectronaut': 'Spectronaut', 'DIA-NN': 'DIANN'}
TICK_FONT, AXIS_TITLE_FONT = dict(family='Arial', size=14), dict(family='Arial', size=18)
LEGEND = dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.7)', font=dict(family='Arial', size=14))
FONT = dict(family='Arial', color='black')
xcat = [str(n) for n in HELA_CELL_ORDER]

fig = go.Figure()
for engine, color in ENGINE_COLOR.items():
    m = depth_mean[engine].reindex(HELA_CELL_ORDER)
    fig.add_trace(go.Scatter(x=xcat, y=m.tolist(), mode='lines+markers', name=ENGINE_LABEL[engine],
                             marker=dict(size=12, color=color, line=dict(width=0.5, color='black')),
                             line=dict(width=3, color=color)))
fig.update_layout(template='plotly_white', width=600, height=600, legend=LEGEND, font=FONT,
                  xaxis_title='Sorted HeLa cells', yaxis_title='Class I phosphosites')
fig.update_xaxes(type='category', categoryorder='array', categoryarray=xcat, tickfont=TICK_FONT, title_font=AXIS_TITLE_FONT)
fig.update_yaxes(tickformat=',', tickfont=TICK_FONT, title_font=AXIS_TITLE_FONT)
fig.show()
fig.write_image(r'figures/figure3/suppl_figure3i_diann_depth.pdf', width=600, height=600)

In [15]:
# Suppl 3j - Class I site overlap between engines (sites pooled over the 3 replicates per cell number):
# % of Spectronaut sites also identified by DIA-NN, and Jaccard index (shared / union).
fig = go.Figure()
for col, label, color, dash in (('pct_spectronaut_in_diann', '% of SN sites identified by DIANN', '#2b8cbe', 'solid'),
                                ('jaccard_pct', 'Jaccard overlap between SN and DIANN', '#6a51a3', 'dash')):
    fig.add_trace(go.Scatter(x=overlap_df['cells'].astype(str).tolist(), y=overlap_df[col].tolist(), mode='lines+markers',
                             name=label, line=dict(width=3, color=color, dash=dash),
                             marker=dict(size=11, color=color, line=dict(width=0.5, color='black'))))
# legend bottom right: at the top left it would cover the 300-1000-cell points
fig.update_layout(template='plotly_white', width=600, height=600, font=FONT,
                  legend=LEGEND | dict(x=0.98, y=0.02, xanchor='right', yanchor='bottom'),
                  xaxis_title='Sorted HeLa cells', yaxis_title='Overlap between DIANN and SN, %')
fig.update_xaxes(type='category', categoryorder='array', categoryarray=xcat, tickfont=TICK_FONT, title_font=AXIS_TITLE_FONT)
fig.update_yaxes(range=[0, 70], tickfont=TICK_FONT, title_font=AXIS_TITLE_FONT)
fig.show()
fig.write_image(r'figures/figure3/suppl_figure3j_diann_overlap.pdf', width=600, height=600)

In [16]:
# Suppl 3k-l - log2 intensity of shared vs engine-specific Class I sites at 3,000 cells, each engine on its
# own quantification scale (k Spectronaut, l DIA-NN). Thousands of sites per group -> no point overlay.
N_SHOW = 3000
Si, Di = engine_sites[N_SHOW]
shared = sorted(set(Si.index) & set(Di.index))
for panel, I, other, only_label, only_color in (('k', Si, Di, 'SN-unique', 'darkred'),
                                               ('l', Di, Si, 'DIANN-unique', '#6a51a3')):
    groups = ((('Shared', I[shared]), '#2b8cbe'),
              ((only_label, I[sorted(set(I.index) - set(other.index))]), only_color))
    fig = go.Figure()
    for (name, vals), color in groups:
        fig.add_trace(go.Violin(y=vals.values, name=name, line=dict(color=color, width=3), box_visible=True,
                                meanline_visible=True, points=False, width=0.8))
    fig.update_layout(template='plotly_white', width=600, height=600, showlegend=False, font=FONT,
                      yaxis_title='log2(intensity)')
    fig.update_xaxes(tickfont=TICK_FONT, title_font=AXIS_TITLE_FONT)
    fig.update_yaxes(tickfont=TICK_FONT, title_font=AXIS_TITLE_FONT)
    fig.show()
    fig.write_image(rf'figures/figure3/suppl_figure3{panel}_diann_abundance.pdf', width=600, height=600)

In [17]:
# === PRIDE MetaInfo export (run after all panels above) ===
import sys; sys.path.insert(0, r"src")
from metainfo_export import dump_panel
SFIG = 3   # figure number (single source of truth for sheet labels)
from core import count_sites_per_sample_ptm_report, process_ptm_site_report
import numpy as np, pandas as pd
def _try(fn, sheet):
    try: fn()
    except Exception as e: print(f"  [SKIP {sheet}] {type(e).__name__}: {e}")

_try(lambda: dump_panel(pd.DataFrame([{"Condition":f"{n}cells","Replicate":i+1,"Phosphopeptide precursors":int(v)}
    for n in sorted(hela_precursors) for i,v in enumerate(hela_precursors[n])]),f"Suppl Figure {SFIG}a"),f"Suppl Figure {SFIG}a")
def _s3c():
    rows=[]
    for n in HELA_CELL_ORDER:
        if n not in hela_precursors or n not in uphos_precursors: continue
        up=list(uphos_precursors[n]); um=float(np.mean(up))
        for i,v in enumerate(hela_precursors[n]):
            rows.append({"Condition":f"{n}cells","Replicate":i+1,"nanoPhos precursors":int(v),
                         "uPhos precursors":int(up[i]) if i<len(up) else None,
                         "uPhos mean precursors":um,"Ratio (nanoPhos / uPhos mean)":v/um})
    dump_panel(pd.DataFrame(rows),f"Suppl Figure {SFIG}c",notes={"crossbar":"mean ± SD of the ratios per cell number"})
_try(_s3c,f"Suppl Figure {SFIG}c")

_try(lambda: dump_panel(pd.concat({f"{n}cells":cv_by_cells[n].reset_index(drop=True) for n in cv_by_cells},axis=1),
    f"Suppl Figure {SFIG}b"),f"Suppl Figure {SFIG}b")
_try(lambda: dump_panel(df_sel,f"Suppl Figure {SFIG}d"),f"Suppl Figure {SFIG}d")
def _s2c():
    rows=[]
    for n in counts_by_cells:
        for rep,v in enumerate(counts_by_cells[n],1):
            rows.append({"Condition":f"{n}cells","Replicate":rep,"Number of class I phosphosites":int(v)})
    dump_panel(pd.DataFrame(rows),f"Suppl Figure {SFIG}e")
_try(_s2c,f"Suppl Figure {SFIG}e")
def _s2d():
    rows=[]
    for n in depth_by_cells:
        for st,vals in depth_by_cells[n].items():
            for rep,v in enumerate(vals,1):
                rows.append({"Cells":n,"State":st,"Replicate":rep,"Number of class I phosphosites":int(v)})
    dump_panel(pd.DataFrame(rows),f"Suppl Figure {SFIG}g")
_try(_s2d,f"Suppl Figure {SFIG}g")
_try(lambda: dump_panel(pd.DataFrame([{"State":s,"Replicate":i+1,"Phosphopeptide precursors":int(v)}
    for s in ["2iL","SL","RA"] for i,v in enumerate(stem500_precursors.get(s,[]))]),f"Suppl Figure {SFIG}f"),f"Suppl Figure {SFIG}f")
_try(lambda: dump_panel(pd.DataFrame([{"Cells":n,"State":s,"Replicate":i+1,"Phosphopeptide precursors":int(v)}
    for n in stem_precursors_by_cells for s,vals in stem_precursors_by_cells[n].items() for i,v in enumerate(vals)]),f"Suppl Figure {SFIG}h"),f"Suppl Figure {SFIG}h")
_try(lambda: dump_panel(depth_df, f"Suppl Figure {SFIG}i"), f"Suppl Figure {SFIG}i")
_try(lambda: dump_panel(overlap_df, f"Suppl Figure {SFIG}j"), f"Suppl Figure {SFIG}j")
def _s3kl():
    Si, Di = engine_sites[N_SHOW]
    for panel, I, other in (("k", Si, Di), ("l", Di, Si)):
        dump_panel(pd.DataFrame({"Site": I.index, "Shared": I.index.isin(other.index),
                                 "Mean log2 intensity": I.values}), f"Suppl Figure {SFIG}{panel}",
                   notes={"cells": N_SHOW, "engine": {"k": "Spectronaut", "l": "DIA-NN"}[panel],
                          "Shared": "site also identified by the other engine"})
_try(_s3kl, f"Suppl Figure {SFIG}k-l")
print(f"Suppl Figure {SFIG} export done.")

  [MetaInfo] wrote 'Suppl Figure 3a'  (18 rows x 3 cols)


  [MetaInfo] wrote 'Suppl Figure 3c'  (18 rows x 6 cols, 1 notes)


  [MetaInfo] wrote 'Suppl Figure 3b'  (11014 rows x 6 cols)


  [MetaInfo] wrote 'Suppl Figure 3d'  (18 rows x 2 cols)


  [MetaInfo] wrote 'Suppl Figure 3e'  (18 rows x 3 cols)


  [MetaInfo] wrote 'Suppl Figure 3g'  (45 rows x 4 cols)


... (2 outputs left out)

  [MetaInfo] wrote 'Suppl Figure 3i'  (36 rows x 4 cols)


  [MetaInfo] wrote 'Suppl Figure 3j'  (6 rows x 6 cols)


  [MetaInfo] wrote 'Suppl Figure 3k'  (21998 rows x 3 cols, 3 notes)


  [MetaInfo] wrote 'Suppl Figure 3l'  (17332 rows x 3 cols, 3 notes)
Suppl Figure 3 export done.
